# Durum Yönetimi (State Management)

Bir Flutter uygulamasının **durumu (state)**, arayüzü (UI) görüntülemek
veya sistem kaynaklarını yönetmek için kullandığı tüm nesneleri ifade
eder. Durum yönetimi, bu nesnelere en etkili şekilde erişmek ve onları
farklı widget’lar arasında paylaşmak için uygulamamızı nasıl organize
ettiğimizdir.

## `StatefulWidget` Kullanımı

Durumu yönetmenin en basit yolu, durumu kendi içinde saklayan bir
`StatefulWidget` kullanmaktır. Örneğin, aşağıdaki widget’ı ele alalım:

``` dart
class MyCounter extends StatefulWidget {
  const MyCounter({super.key});

  @override
  State<MyCounter> createState() => _MyCounterState();
}

class _MyCounterState extends State<MyCounter> {
  int count = 0;

  @override
  Widget build(BuildContext context) {
    return Column(
      children: [
        Text('Count: $count'),
        TextButton(
          onPressed: () {
            setState(() {
              count++;
            });
          },
          child: Text('Increment'),
        )
      ],
    );
  }
}
```

Bu kod, durum yönetimi hakkında düşünürken iki önemli kavramı gösterir:

- **Kapsülleme (Encapsulation):** `MyCounter` kullanan widget, alttaki
  `count` değişkenini göremez ve ona erişmek veya değiştirmek için
  hiçbir araca sahip değildir.
- **Nesne yaşam döngüsü (Object lifecycle):** `_MyCounterState` nesnesi
  ve onun `count` değişkeni, `MyCounter` ilk kez oluşturulduğunda
  yaratılır ve ekrandan kaldırılana kadar var olur. Bu, **geçici durum
  (ephemeral state)** örneğidir.

## Widget’lar Arasında Durum Paylaşımı

Bir uygulamanın durumu depolaması gereken bazı senaryolar şunlardır:

- Paylaşılan durumu **güncellemek** ve uygulamanın diğer kısımlarını
  bilgilendirmek.
- Paylaşılan durumdaki değişiklikleri **dinlemek** ve değiştiğinde
  arayüzü yeniden oluşturmak.

Bu bölüm, uygulamanızdaki farklı widget’lar arasında durumu nasıl etkili
bir şekilde paylaşabileceğinizi inceler. En yaygın modeller şunlardır:

- **Widget yapıcılarını kullanmak** (diğer framework’lerde bazen “prop
  drilling” olarak adlandırılır).
- **`InheritedWidget` kullanmak** (veya `provider` paketi gibi benzer
  bir API).
- Bir üst widget’a bir şeyin değiştiğini bildirmek için **geri
  çağırmaları (callbacks) kullanmak**.

### Widget Yapıcılarını Kullanmak

Dart nesneleri referansla iletildiği için, widget’ların ihtiyaç
duydukları nesneleri yapıcılarında (constructor) tanımlamaları çok
yaygındır. Bir widget’ın yapıcısına ilettiğiniz herhangi bir durum,
arayüzünü oluşturmak için kullanılabilir:

``` dart
class MyCounter extends StatelessWidget {
  final int count;
  const MyCounter({super.key, required this.count});

  @override
  Widget build(BuildContext context) {
    return Text('$count');
  }
}
```

Bu, widget’ınızı kullanan diğer kişilerin, onu kullanmak için ne
sağlamaları gerektiğini bilmelerini açık hale getirir:

``` dart
Column(
  children: [
    MyCounter(
      count: count,
    ),
    MyCounter(
      count: count,
    ),
    TextButton(
      child: Text('Increment'),
      onPressed: () {
        setState(() {
          count++;
        });
      },
    )
  ],
)
```

Uygulamanız için paylaşılan verileri widget yapıcıları aracılığıyla
iletmek, kodu okuyan herkes için paylaşılan bağımlılıklar olduğunu
netleştirir. Sadece verileri çocuklarına iletmek için bir widget zinciri
oluşturmak genellikle **prop drilling** olarak adlandırılır.

### `InheritedWidget` Kullanmak

Verileri widget ağacında manuel olarak aşağı iletmek zahmetli olabilir
ve istenmeyen kod tekrarına (boilerplate) neden olabilir. Flutter,
verileri bir ebeveyn widget’ta verimli bir şekilde barındırmanın ve
çocuk widget’ların bunları bir alan olarak saklamadan erişebilmesinin
bir yolunu sağlayan `InheritedWidget`’ı sunar.

`InheritedWidget` kullanmak için, `InheritedWidget` sınıfını genişletin
(extend) ve `dependOnInheritedWidgetOfExactType` kullanarak statik
`of()` metodunu uygulayın. Bir `build` metodunda `of()` çağıran bir
widget, Flutter framework tarafından yönetilen bir bağımlılık oluşturur.
Böylece bu widget yeni verilerle yeniden oluşturulduğunda ve
`updateShouldNotify` true döndürdüğünde, bu `InheritedWidget`’a bağlı
olan tüm widget’lar yeniden oluşturulur.

``` dart
class MyState extends InheritedWidget {
  const MyState({
    super.key,
    required this.data,
    required super.child,
  });

  final String data;

  static MyState of(BuildContext context) {
    // This method looks for the nearest `MyState` widget ancestor.
    final result = context.dependOnInheritedWidgetOfExactType<MyState>();

    assert(result != null, 'No MyState found in context');

    return result!;
  }

  @override
  // This method should return true if the old widget's data is different
  // from this widget's data. If true, any widgets that depend on this widget
  // by calling `of()` will be re-built.
  bool updateShouldNotify(MyState oldWidget) => data != oldWidget.data;
}
```

Ardından, paylaşılan duruma erişmesi gereken widget’ın `build()`
metodundan `of()` metodunu çağırın:

``` dart
class HomeScreen extends StatelessWidget {
  const HomeScreen({super.key});

  @override
  Widget build(BuildContext context) {
    var data = MyState.of(context).data;
    return Scaffold(
      body: Center(
        child: Text(data),
      ),
    );
  }
}
```

### Geri Çağrılar (Callbacks)

Bir değer değiştiğinde bir geri çağırma (callback) sunarak diğer
widget’ları bilgilendirebilirsiniz. Flutter, tek parametreli bir
fonksiyon geri çağırması bildiren `ValueChanged` türünü sağlar:

``` dart
typedef ValueChanged<T> = void Function(T value);
```

Widget’ınızın yapıcısında `onChanged` özelliğini sunarak, bu widget’ı
kullanan herhangi bir widget’ın bir değer değiştiğinde tepki vermesini
sağlarsınız:

``` dart
class MyCounter extends StatefulWidget {
  const MyCounter({super.key, required this.onChanged});

  final ValueChanged<int> onChanged;

  @override
  State<MyCounter> createState() => _MyCounterState();
}
```

Örneğin, bu widget `onPressed` callback’ini yönetebilir ve  
`count` değişkeninin en güncel iç durumunu `onChanged` fonksiyonu
aracılığıyla iletebilir.

``` dart
TextButton(
  onPressed: () {
    widget.onChanged(count++);
  },
),
```

## Dinlenebilirleri Kullanmak (Using Listenables)

`Listenable`, değerlerinde değişiklik olduğunda dinleyicileri
(listeners) bilgilendiren nesneler için soyut bir sınıftır.

### `ChangeNotifier`

`ChangeNotifier`, Flutter’da durum yönetimi için kullanılan basit bir
sınıftır. `Listenable`’ı genişletir ve bir şeyler değiştiğinde
dinleyicilerine bildirim gönderir.

``` dart
class CounterNotifier extends ChangeNotifier {
  int _count = 0;
  int get count => _count;

  void increment() {
    _count++;
    notifyListeners();
  }
}
```

UI’da bir `ChangeNotifier`’ı dinlemek için, `ListenableBuilder`
widget’ını kullanabilir veya `addListener` metodunu kullanarak
dinleyiciyi manuel olarak ekleyebilirsiniz.

``` dart
Column(
  children: [
    ListenableBuilder(
      listenable: counterNotifier,
      builder: (context, child) {
        return Text('counter: ${counterNotifier.count}');
      },
    ),
    TextButton(
      child: Text('Increment'),
      onPressed: () {
        counterNotifier.increment();
      },
    ),
  ],
)
```

### `ValueNotifier`

`ValueNotifier`, tek bir değer saklayan, `ChangeNotifier`’ın daha basit
bir sürümüdür.  
`ValueListenable` ve `Listenable` arayüzlerini uygular; bu sayede
`ListenableBuilder` ve `ValueListenableBuilder` gibi widget’larla
uyumludur.

Kullanmak için, başlangıç değeri ile bir `ValueNotifier` örneği
oluşturmanız yeterlidir:

``` dart
final ValueNotifier<int> _counter = ValueNotifier<int>(0);
```

Ardından `value` alanını kullanarak değeri okuyabilir veya
güncelleyebilir ve  
değerin değiştiğini tüm dinleyicilere (listeners) bildirebilirsiniz.

`ValueNotifier`, `ChangeNotifier`’dan türediği için aynı zamanda bir
`Listenable`’dır  
ve `ListenableBuilder` ile birlikte kullanılabilir.

Ancak ayrıca `ValueListenableBuilder` da kullanabilirsiniz.  
Bu widget, builder callback fonksiyonu içinde değeri doğrudan sağlar:

``` dart
Column(
  children: [
    ValueListenableBuilder(
      valueListenable: counterNotifier,
      builder: (context, value, child) {
        return Text('counter: $value');
      },
    ),
    TextButton(
      child: Text('Increment'),
      onPressed: () {
        counterNotifier.value++;
      },
    ),
  ],
)
```

## Uygulama Mimarisi için MVVM Kullanımı

Daha karmaşık uygulamalar için, durum yönetimi mantığınızı UI kodunuzdan
ayırmak üzere bir mimari desen kullanmayı düşünebilirsiniz. Flutter’daki
yaygın bir desen **Model-View-ViewModel (MVVM)**’dir.

- **Model:** Uygulamanızın verilerini ve iş mantığını temsil eder.
  UI’dan veya nasıl görüntülendiğinden habersizdir.
- **View:** Kullanıcı arayüzünü (UI) temsil eder. Kullanıcı
  etkileşimlerini dinler ve ViewModel’e iletir. Ayrıca ViewModel’i
  dinler ve durum değiştiğinde kendini günceller.
- **ViewModel:** View ve Model arasında bir köprü görevi görür. Modelden
  verileri alır ve View’ın görüntüleyebileceği bir formata dönüştürür.
  Ayrıca View’dan gelen kullanıcı etkileşimlerini işler ve Modeli
  günceller.

### Model Tanımlama

Model, genellikle HTTP istekleri yapmak, verileri önbelleğe almak veya
bir eklenti (plugin) gibi sistem kaynaklarını yönetmek gibi düşük
seviyeli görevleri yerine getiren bir Dart sınıfıdır. Bir modelin
genellikle Flutter kütüphanelerini içe aktarması (import) gerekmez.

Örneğin, bir HTTP istemcisi kullanarak sayaç durumunu yükleyen veya
güncelleyen bir modeli düşünün:

``` dart
import 'package:http/http.dart';

class CounterData {
  CounterData(this.count);

  final int count;
}

class CounterModel {
  Future<CounterData> loadCountFromServer() async {
    final uri = Uri.parse('https://myfluttercounterapp.net/count');
    final response = await get(uri);

    if (response.statusCode != 200) {
      throw ('Failed to update resource');
    }

    return CounterData(int.parse(response.body));
  }

  Future<CounterData> updateCountOnServer(int newCount) async {
    // ...
  }
}
```

Bu model herhangi bir Flutter ilkelini (primitives) kullanmaz veya
çalıştığı platform hakkında herhangi bir varsayımda bulunmaz; tek görevi
HTTP istemcisini kullanarak sayacı getirmek veya güncellemektir. Bu
durum, modelin birim testlerinde (unit tests) bir Mock veya Fake ile
uygulanmasına olanak tanır ve uygulamanızın düşük seviyeli bileşenleri
ile tam uygulamayı oluşturmak için gereken daha yüksek seviyeli UI
bileşenleri arasında net sınırlar tanımlar.

`CounterData` sınıfı verinin yapısını tanımlar ve uygulamamızın gerçek
“model”idir. Model katmanı genellikle uygulamanız için gereken temel
algoritmalardan ve veri yapılarından sorumludur. Eğer modeli tanımlamak
için değişmez (immutable) değer türlerini kullanmak gibi başka yollarla
ilgileniyorsanız, pub.dev üzerindeki `freezed` veya `build_collection`
gibi paketlere göz atın.

### Görünüm Modeli (ViewModel) Tanımlama

Bir `ViewModel`, `View`’ı `Model`’e bağlar. Modeli, View tarafından
doğrudan erişilmekten korur ve veri akışının modelde yapılan bir
değişiklikle başlamasını sağlar. Veri akışı, View’a bir şeylerin
değiştiğini bildirmek için `notifyListeners` kullanan `ViewModel`
tarafından yönetilir. `ViewModel`, mutfak (model) ile müşteriler
(view’lar) arasındaki iletişimi sağlayan bir restorandaki garson
gibidir.

``` dart
import 'package:flutter/foundation.dart';

class CounterViewModel extends ChangeNotifier {
  final CounterModel model;
  int? count;
  String? errorMessage;
  CounterViewModel(this.model);

  Future<void> init() async {
    try {
      count = (await model.loadCountFromServer()).count;
    } catch (e) {
      errorMessage = 'Could not initialize counter';
    }
    notifyListeners();
  }

  Future<void> increment() async {
    final currentCount = count;
    if (currentCount == null) {
      throw('Not initialized');
    }
    try {
      final incrementedCount = currentCount + 1;
      await model.updateCountOnServer(incrementedCount);
      count = incrementedCount;
    } catch(e) {
      errorMessage = 'Could not update count';
    }
    notifyListeners();
  }
}
```

`ViewModel`’in Modelden bir hata aldığında bir `errorMessage`
sakladığına dikkat edin. Bu, View’ı çökmeye yol açabilecek işlenmemiş
çalışma zamanı hatalarından korur. Bunun yerine, `errorMessage` alanı
view tarafından kullanıcı dostu bir hata mesajı göstermek için
kullanılabilir.

### Görünümü (View) Tanımlama

`ViewModel`imiz bir `ChangeNotifier` olduğundan, ona referansı olan
herhangi bir widget, `ViewModel` dinleyicilerini bilgilendirdiğinde
widget ağacını yeniden oluşturmak için bir `ListenableBuilder`
kullanabilir:

``` dart
ListenableBuilder(
  listenable: viewModel,
  builder: (context, child) {
    return Column(
      children: [
        if (viewModel.errorMessage != null)
          Text(
            'Error: ${viewModel.errorMessage}',
            style: Theme.of(context)
                .textTheme
                .labelSmall
                ?.apply(color: Colors.red),
          ),
        Text('Count: ${viewModel.count}'),
        TextButton(
          onPressed: () {
            viewModel.increment();
          },
          child: Text('Increment'),
        ),
      ],
    );
  },
)
```

Bu desen, uygulamanızın iş mantığının UI mantığından ve Model katmanı
tarafından gerçekleştirilen düşük seviyeli işlemlerden ayrı olmasını
sağlar.

## 📄 Lisans Bilgisi

Bu doküman, **Flutter resmi dokümantasyonundan** türetilmiş Türkçe ders
notudur.

**Orijinal kaynak:**  
https://docs.flutter.dev/data-and-backend/state-mgmt

**Türkçe çeviri ve düzenleme:**  
[Doç. Dr. Hakan Temiz](mailto:htemiz@artvin.edu.tr)

------------------------------------------------------------------------

### Lisans Kapsamı

Bu dokümandaki içerikler aşağıdaki açık lisanslar kapsamında
sunulmaktadır:

**Metin içerikleri (anlatım ve açıklamalar):**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** Creative Commons Attribution 4.0 International (CC BY 4.0)  
https://creativecommons.org/licenses/by/4.0/

Bu lisans kapsamında: - İçerik kopyalanabilir, dağıtılabilir ve
uyarlanabilir  
- Ticari kullanım serbesttir  
- Kaynak belirtilmesi zorunludur

**Kod örnekleri:**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** BSD 3-Clause License  
https://opensource.org/licenses/BSD-3-Clause

Bu lisans kapsamında: - Kodlar kopyalanabilir, değiştirilebilir ve
dağıtılabilir  
- Ticari kullanım serbesttir  
- Lisans bildiriminin korunması gerekir